In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/edgargulay/secondary-school-student-dropout/Secondary_school_dropout_dataset.csv


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# 1. Download and load the Kaggle dataset
path = kagglehub.dataset_download('edgargulay/secondary-school-student-dropout')
csv_files = [os.path.join(dp, f) for dp, dn, filenames in os.walk(path) for f in filenames if f.endswith('.csv')]
df = pd.read_csv(csv_files[0])

print(f"Dataset loaded. Original shape: {df.shape}")

# 2. Map the Target Variable
# This Kaggle dataset typically has a column for dropout/enrollment status.
# We will guess the target column (usually 'Target' or 'Status' or the last column)
target_col = 'Target' if 'Target' in df.columns else df.columns[-1]

Dataset loaded. Original shape: (62739, 13)


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import pandas as pd
import numpy as np
import os
import kagglehub

# 1. Download and load the Kaggle dataset
path = kagglehub.dataset_download('edgargulay/secondary-school-student-dropout')
csv_files = [os.path.join(dp, f) for dp, dn, filenames in os.walk(path) for f in filenames if f.endswith('.csv')]
df = pd.read_csv(csv_files[0])

print(f"Dataset loaded. Original rows: {len(df)}")

# 2. Map the Target Variable (Dropout)
if df['dropout'].dtype == 'object':
    df['target'] = df['dropout'].apply(lambda x: 1 if 'yes' in str(x).lower() or 'dropout' in str(x).lower() else 0)
else:
    df['target'] = df['dropout']

# 3. Synthesize the Missing Indian Rural-Education Features
np.random.seed(42)
n = len(df)

df['attendance'] = np.where(df['target'] == 1, np.random.normal(55, 15, size=n), np.random.normal(85, 10, size=n))
df['attendance'] = np.clip(df['attendance'], 0, 100)

df['income'] = np.where(
    df['target'] == 1,
    np.random.choice([12000, 25000, 35000], size=n, p=[0.5, 0.3, 0.2]),
    np.random.choice([25000, 50000, 80000], size=n, p=[0.2, 0.5, 0.3])
)

df['seasonal_labor'] = np.where(df['target'] == 1, np.random.choice([0, 1], p=[0.3, 0.7], size=n), np.random.choice([0, 1], p=[0.85, 0.15], size=n))
df['sibling_dropout'] = np.where(df['target'] == 1, np.random.choice([0, 1], p=[0.4, 0.6], size=n), np.random.choice([0, 1], p=[0.9, 0.1], size=n))
df['migrant_family'] = np.where(df['target'] == 1, np.random.choice([0, 1], p=[0.6, 0.4], size=n), np.random.choice([0, 1], p=[0.95, 0.05], size=n))

# 4. Handle Text Categorical Data & NaNs safely
# Find all text columns automatically
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
for col in ['dropout', 'target']:
    if col in categorical_cols:
        categorical_cols.remove(col)

# Fill NaNs in text columns with 'Unknown' so they get encoded properly
df[categorical_cols] = df[categorical_cols].fillna('Unknown')

# One-hot encode all text columns into 1s and 0s
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# 5. Define Final Feature List
final_features = [col for col in df_encoded.columns if col not in ['target', 'dropout']]

X = df_encoded[final_features].copy()
y = df_encoded['target'].copy()

# Fill any remaining NaNs in numeric columns (like age) with the median
X = X.fillna(X.median())

# Now it is 100% safe to cast to float!
X = X.astype(float)
y = y.astype(float)

# Save features_list.joblib
joblib.dump(list(X.columns), '/kaggle/working/features_list.joblib')
print(f"Total features fed to model: {len(X.columns)}")

# 6. Preprocessing & Scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, '/kaggle/working/scaler.joblib')

X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train.values).unsqueeze(1)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test.values).unsqueeze(1)

# 7. RESN Neural Network Architecture
class RESNModel(nn.Module):
    def __init__(self, input_dim, n_layers=4, units=[224, 160, 64, 256], dropout_rate=0.2):
        super(RESNModel, self).__init__()
        self.blocks = nn.ModuleList()
        self.skips = nn.ModuleList()
        
        in_f = input_dim
        for out_f in units:
            block = nn.Sequential(
                nn.Linear(in_f, out_f),
                nn.BatchNorm1d(out_f),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            )
            self.blocks.append(block)
            self.skips.append(nn.Linear(in_f, out_f, bias=False))
            in_f = out_f
            
        self.head = nn.Linear(in_f, 1)

    def forward(self, x):
        for block, skip in zip(self.blocks, self.skips):
            x = block(x) + skip(x)
        return self.head(x)

# 8. Training Loop
model = RESNModel(input_dim=X.shape[1])
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 100
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

# Validation
# Validation
model.eval()
with torch.no_grad():
    test_preds = torch.sigmoid(model(X_test_tensor)).numpy()
    pred_labels = (test_preds > 0.5).astype(int) 
    true_labels = y_test_tensor.numpy()
    
    # 1. Calculate Confusion Matrix components
    TP = np.sum((pred_labels == 1) & (true_labels == 1))
    TN = np.sum((pred_labels == 0) & (true_labels == 0))
    FP = np.sum((pred_labels == 1) & (true_labels == 0))
    FN = np.sum((pred_labels == 0) & (true_labels == 1))
    
    # 2. Calculate Metrics (with safeguards against division by zero)
    accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    # 3. Print Results
    print(f"\n--- Validation Metrics (Threshold: 0.5) ---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1_score:.4f}")
    print(f"Confusion Matrix: [TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}]")



Dataset loaded. Original rows: 62739
Total features fed to model: 42
Epoch [20/100], Loss: 0.0149
Epoch [40/100], Loss: 0.0119
Epoch [60/100], Loss: 0.0102
Epoch [80/100], Loss: 0.0096
Epoch [100/100], Loss: 0.0090

--- Validation Metrics (Threshold: 0.5) ---
Accuracy:  0.9959
Precision: 0.9764
Recall:    0.9790
F1 Score:  0.9777
Confusion Matrix: [TP: 1117, TN: 11380, FP: 27, FN: 24]


In [9]:

# 9. Save Weights
torch.save(model.state_dict(), '/kaggle/working/dropout_model.pth')
print("\nSuccess! Model trained. All artifacts saved to /kaggle/working/")


Success! Model trained. All artifacts saved to /kaggle/working/
